# 📘 **NOAI 2026 - Programming Task 3**
* **Subject:** Deep Learning (NLP / Transformers)
* **Title:** The Attention Mechanism (Building the "Brain")
* **Total** = 35 Marks
  * **Part 1: The Core Math (10 Marks)**
  * **Part 2: The "Committee" (10 Marks)**
  * **Part 3: The Timekeeper (5 Marks)**
  * **Part 4: The Decision (5 Marks)**
  * **Part 5: Bonus Question (5 Marks)**

Timestamp: 20 Feb 2026

## **1. The Scenario**
You are the Lead Architect at *LinguaTech*. Your team is building a lightweight Language Model to analyze ancient text fragments.

Standard Neural Networks read words one by one and forget the beginning by the time they reach the end. To fix this, you must implement the **Transformer Architecture**, specifically the mechanism that allows the model to look at *all words simultaneously* and understand context.

This mechanism is called **Attention**.

### **The Analogy: The "Search Engine"**
Think of Attention as a database lookup:
1.  **Query ($Q$):** What you are looking for (e.g., "Bank").
2.  **Key ($K$):** The labels in the database (e.g., "River", "Money").
3.  **Value ($V$):** The actual meaning/content.
4.  **Attention Score:** How much the Query matches the Key. If "Bank" appears near "Water", the score for "River" goes up.

### **📚 Supplementary Reading**

* **[The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)**

## **2. The Architecture**
You will build the four pillars of the Transformer Encoder:
1.  **Scaled Dot-Product Attention:** The math that calculates relevance.
2.  **Multi-Head Attention:** Running multiple attention checks in parallel.
3.  **Positional Encoding:** Adding "timestamps" so the model knows word order.
4.  **Classification Head:** Making a final decision based on the data.

In [ ]:
# --- CELL 1: SETUP (DO NOT EDIT) ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# Set seed for reproducibility
torch.manual_seed(2026)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## **Part 1: The Core Math (10 Marks)**
---
### **Question 3.1 Scaled Dot-Product Attention (8 Marks)**
The heart of the Transformer is this formula:

$\displaystyle \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

**Your Task:**
Implement this function using PyTorch.
1.  **MatMul:** Multiply Query ($Q$) and Key Transposed ($K^T$).
2.  **Scale:** Divide by the square root of the dimension ($\sqrt{d_k}$).
3.  **Mask (Optional):** If a mask is provided, use `masked_fill` to set the masked positions to a large negative number like `-1e9` (so Softmax makes them zero).<br>
    i.  Do not use `-inf` as it can cause numerical instability.
4.  **Softmax:** Apply softmax to get probabilities (scores must sum to 1).
5.  **Weighted Sum:** Multiply the scores by Value ($V$).

### Question 3.1 (8 points)

In [ ]:
### ANSWER QUESTION 3.1 HERE


def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Compute scaled dot-product attention.
    
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    
    Parameters:
    -----------
    query : torch.Tensor, shape (batch, num_heads, seq_len, d_k)
        Query vectors
    key : torch.Tensor, shape (batch, num_heads, seq_len, d_k)
        Key vectors
    value : torch.Tensor, shape (batch, num_heads, seq_len, d_v)
        Value vectors
    mask : torch.Tensor, optional, shape (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
        Attention mask (True/1 = keep, False/0 = mask out)
    dropout : nn.Dropout, optional
        Dropout layer for attention weights
        
    Returns:
    --------
    output : torch.Tensor, shape (batch, num_heads, seq_len, d_v)
        Attention output
    attention_weights : torch.Tensor, shape (batch, num_heads, seq_len, seq_len)
        Attention weight matrix
    """
    d_k = query.size(-1)
    
    ### QUESTION 3.1 START

    # TODO Step 1: Compute attention scores: QK^T

    # TODO Step 2: Scale by sqrt(d_k)

    # TODO Step 3: Apply mask if provided

    # TODO Step 4: Apply softmax to get attention weights

    # TODO Step 5: Apply dropout if provided

    # TODO Step 6: Compute output: attention_weights @ V


    ### QUESTION 3.1 END

    return output, attention_weights

In [ ]:
### TEST FOR QUESTION 3.1


# Define test variables
batch = 2
heads = 8
seq_len = 10
d_k = 64

Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

print("Test variables initialized:")
print(f"  Q shape: {Q.shape}")
print(f"  K shape: {K.shape}")
print(f"  V shape: {V.shape}")
print()

print("=" * 60)
print("Testing Scaled Dot-Product Attention...")
print("=" * 60)

try:
    out, attn = scaled_dot_product_attention(Q, K, V)
    print(f"  Output shape: {out.shape}")
    print(f"  Attention shape: {attn.shape}")
    print(f"  Attention sums to 1: {attn.sum(-1).mean().item():.4f}")
    assert out.shape == (batch, heads, seq_len, d_k)
    assert attn.shape == (batch, heads, seq_len, seq_len)
    assert torch.allclose(attn.sum(-1), torch.ones(batch, heads, seq_len), atol=1e-5)
    print("  ✓ Attention working!")
except Exception as e:
    print(f"  ✗ Attention failed: {e}")


# Test with mask
print("\nTesting Attention with Mask...")
mask = torch.ones(batch, 1, 1, seq_len)
mask[:, :, :, -3:] = 0  # Mask last 3 positions
try:
    out_masked, attn_masked = scaled_dot_product_attention(Q, K, V, mask=mask)
    # Check that masked positions have zero attention
    assert (attn_masked[:, :, :, -3:] == 0).all(), "Masked positions should have 0 attention"
    print("  ✓ Masked attention working!")
except Exception as e:
    print(f"  ✗ Masked attention failed: {e}")

<!-- BEGIN QUESTION -->

### Question 3.2 (2 points)

Why is scaling by `sqrt(d_k)` necessary?

In [ ]:
### ANSWER QUESTION 3.2 HERE

"""
Why is scaling by sqrt(d_k) necessary?

Your answer should explain (2-4 sentences):
1. What happens to dot product magnitudes as d_k increases
2. How this affects softmax behavior (hint: saturation)
3. Why this causes gradient flow problems

Example structure:
"When d_k is large (e.g., 512), dot products between Q and K become very large in magnitude.
This causes softmax to... which leads to... This is problematic for training because..."
"""

answer_3_2 = """
TYPE YOUR ANSWER HERE:


"""

print(answer_3_2)

<!-- END QUESTION -->

## **Part 2: The "Committee" (10 Marks)**
---
### **Question 3.3 & 3.4 Multi-Head Attention**
One attention head is not enough. It might focus on grammar, but miss the tone.
**Multi-Head Attention** splits the input into $h$ different "heads," allowing the model to look at different aspects of the sentence simultaneously.

**The Logic:**
1.  **Split:** Divide the input embeddings into `num_heads` smaller chunks.
2.  **Calculate:** Run Scaled Dot-Product Attention on all chunks in parallel.
3.  **Concatenate:** Glue the results back together.
4.  **Project:** Pass through a final Linear layer.
---

### Question 3.3 (5 points) & 3.4 (5 points)

In [ ]:
### ANSWER QUESTION 3.3 & 3.4 HERE


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism.
    
    MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O
    where head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)
    """
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Parameters:
        -----------
        d_model : int
            Model dimension (must be divisible by num_heads)
        num_heads : int
            Number of attention heads
        dropout : float
            Dropout probability
        """
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        ### QUESTION 3.3 START
        # Define projection layers:
        # - W_q: for queries
        # - W_k: for keys
        # - W_v: for values
        # - W_o: for output projection
        # - dropout layer
        
        
        ### QUESTION 3.3 END
    
    def forward(self, query, key, value, mask=None):
        """
        Parameters:
        -----------
        query : torch.Tensor, shape (batch, seq_len, d_model)
        key : torch.Tensor, shape (batch, seq_len, d_model)
        value : torch.Tensor, shape (batch, seq_len, d_model)
        mask : torch.Tensor, optional
        
        Returns:
        --------
        output : torch.Tensor, shape (batch, seq_len, d_model)
        attention_weights : torch.Tensor, shape (batch, num_heads, seq_len, seq_len)
        """
        batch_size = query.size(0)
        
        ### QUESTION 3.4 START
        # 1. Apply linear projections to get Q, K, V
        # 2. Reshape for multi-head
        # 3. Apply scaled_dot_product_attention
        # 4. Reshape back
        # 5. Apply output projection W_o
        
        ### QUESTION 3.4 END

In [ ]:
### TEST FOR QUESTION 3.3 & 3.4


# Ensure test variables are defined
if 'batch' not in locals() or 'seq_len' not in locals():
    batch = 2
    seq_len = 10
    print("Note: Using default batch=2, seq_len=10 for testing")

print("=" * 60)
print("Testing Multi-Head Attention...")
print("=" * 60)
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(batch, seq_len, 512)

try:
    out, attn = mha(x, x, x)
    print(f"  Input shape: {x.shape}")
    print(f"  Output shape: {out.shape}")
    print(f"  Attention shape: {attn.shape}")
    assert out.shape == x.shape
    print("  ✓ Multi-Head Attention working!")
except Exception as e:
    print(f"  ✗ Multi-Head Attention failed: {e}")

## **Part 3: The Timekeeper (5 Marks)**
---
### **Question 3.5 Debugging Positional Encoding (5 Marks)**
Transformers process all words at the same time (parallel). This makes them fast, but it means **they don't know word order**. To the model, *"The dog bit the man"* is the same as *"The man bit the dog"*.

To fix this, we add a **Positional Encoding** vector to the word embeddings.

**The Bug:**
A junior engineer wrote the class below, but the sine/cosine wavelengths are messed up.
**Your Task:** Identify the indexing error in the `forward` or `__init__` logic and fix it so that even and odd indices get the correct trigonometric functions.

---

In [ ]:
### QUESTION 3.5 START: Fix the Bug in Positional Encoding (10 pts) ###


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create matrix of [SeqLen, HiddenDim] representing the positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # --- THE BUG IS IN THE LINE BELOW ---
        # 
        # The correct formula for positional encoding is:
        #   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
        #   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
        #
        # The divisor term should be: 1 / 10000^(2i/d_model)
        # Using logarithms: 1 / 10000^x = exp(-x * log(10000))
        # 
        # The bug is in THIS line - the formula is wrong:
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        # TODO: Fix the div_term calculation to match the formula above
        # Hint: What should be inside the parentheses after the * operator?
        # -----------------------

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [Batch, SeqLen, HiddenDim]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

### QUESTION 3.5 END ###

In [ ]:
### TEST FOR QUESTION 3.5


print("=" * 60)
print("Testing Positional Encoding...")
print("=" * 60)
pe = PositionalEncoding(d_model=512)
x = torch.randn(2, 100, 512)

try:
    out = pe(x)
    print(f"  Input shape: {x.shape}")
    print(f"  Output shape: {out.shape}")
    assert out.shape == x.shape
    
    # Test that PE is added correctly
    pe_no_dropout = PositionalEncoding(d_model=512, dropout=0.0)
    pe_no_dropout.eval()
    x_test = torch.zeros(1, 10, 512)
    out_test = pe_no_dropout(x_test)
    # Output should equal just the positional encoding
    assert not torch.allclose(out_test, x_test), "PE should add non-zero values"
    
    # Test GPU transfer
    pe_cuda = PositionalEncoding(d_model=512).to(device)
    x_cuda = torch.randn(2, 100, 512).to(device)
    out_cuda = pe_cuda(x_cuda)
    print(f"  GPU test passed: output on {out_cuda.device}")
    print("  ✓ Positional Encoding working!")
except Exception as e:
    print(f"  ✗ Positional Encoding failed: {e}")

In [ ]:
# --- NUMERICAL CORRECTNESS TEST FOR ATTENTION ---

print("\n" + "=" * 60)
print("Testing Attention Numerical Correctness...")
print("=" * 60)

# Create simple test case
batch_test = 1
heads_test = 1
seq_test = 3
d_k_test = 4

Q_test = torch.ones(batch_test, heads_test, seq_test, d_k_test)
K_test = torch.ones(batch_test, heads_test, seq_test, d_k_test)  
V_test = torch.arange(12).reshape(batch_test, heads_test, seq_test, d_k_test).float()

try:
    out_test, attn_test = scaled_dot_product_attention(Q_test, K_test, V_test)
    
    # Check attention weights sum to 1
    attn_sums = attn_test.sum(dim=-1)
    assert torch.allclose(attn_sums, torch.ones_like(attn_sums), atol=1e-5), \
        "Attention weights must sum to 1 across seq_len dimension"
    
    # Check output shape
    assert out_test.shape == V_test.shape, \
        f"Output shape {out_test.shape} doesn't match Value shape {V_test.shape}"
    
    print("  ✓ Attention weights sum to 1")
    print("  ✓ Output shape correct")
    print("  ✓ Numerical correctness verified!")
    
except AssertionError as e:
    print(f"  ❌ Assertion failed: {e}")
except Exception as e:
    print(f"  ❌ Test failed: {e}")

## **Part 4: The Decision (5 Marks)**
---
### **Question 3.6 & 3.7 The Classification Head (5 Marks)**
Now that the Transformer Encoder has processed the text and understood the context, we need to make a prediction (e.g., Is this sentence Positive or Negative?).

**Your Task:**
Implement a simple classifier that:
- Takes the **mean** (average) of the encoder outputs **over the sequence dimension** (dim=1).
- Passes the result through a fully connected layer.
---

### Question 3.6 (2 points) & 3.7 (3 points)

Transformer Encoder for Classification

In [ ]:
class TransformerEncoderLayer(nn.Module):
    """Single Transformer encoder layer."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection and layer norm
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward with residual connection and layer norm
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))
        
        return x


class TransformerClassifier(nn.Module):
    """Transformer encoder for text classification."""
    
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=4, 
                 d_ff=1024, num_classes=2, max_len=512, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        
        # Embedding layers
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Transformer encoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        ### QUESTION 3.6 START
        # Define classification head:
        # Option 1: Use [CLS] token representation
        # Option 2: Use mean pooling over sequence
        # Include: LayerNorm, Linear(d_model, d_model), ReLU, Dropout, Linear(d_model, num_classes)
        
        
        ### QUESTION 3.6 END
    
    def forward(self, input_ids, attention_mask=None):
        """
        Parameters:
        -----------
        input_ids : torch.Tensor, shape (batch, seq_len)
            Token indices
        attention_mask : torch.Tensor, shape (batch, seq_len)
            Mask for padding tokens (1 = real token, 0 = padding)
            
        Returns:
        --------
        logits : torch.Tensor, shape (batch, num_classes)
        """
        ### QUESTION 3.7 START
        # 1. Get token embeddings and scale by sqrt(d_model)
        # 2. Add positional encoding
        # 3. Convert attention_mask to proper format for self-attention
        # 4. Pass through encoder layers
        # 5. Apply classification head (pool sequence, then classify)
        
        
        ### QUESTION 3.7 END

In [ ]:
### TEST FOR QUESTION 3.6 & 3.7


print("=" * 60)
print("Testing Transformer Classifier...")
print("=" * 60)
vocab_size = 30000
model = TransformerClassifier(
    vocab_size=vocab_size,
    d_model=256,
    num_heads=heads,
    num_layers=4,
    num_classes=2
).to(device)

input_ids = torch.randint(0, vocab_size, (batch, seq_len)).to(device)
attention_mask = torch.ones(batch, seq_len).to(device)
attention_mask[:, -2:] = 0  # Last 2 tokens are padding

try:
    logits = model(input_ids, attention_mask)
    print(f"  Input shape: {input_ids.shape}")
    print(f"  Logits shape: {logits.shape}")
    print(f"  Predictions: {logits.argmax(-1).tolist()}")
    assert logits.shape == (2, 2)
    print("  ✓ Transformer Classifier working!")
    
    # Test without mask
    logits_no_mask = model(input_ids)
    print(f"  Without mask - Logits shape: {logits_no_mask.shape}")
    print("  ✓ Works without mask too!")
except Exception as e:
    print(f"  ✗ Transformer Classifier failed: {e}")

In [ ]:
### ANSWER BONUS QUESTION 3.8 START


def create_causal_mask(seq_len, device='cpu'):
    """
    Create a causal (lower triangular) mask for decoder self-attention.
    
    Why needed: In language models, when generating token at position i,
    the model should only attend to tokens at positions 0 to i (not future tokens).
    This prevents "cheating" during training.
    
    Example for seq_len=4:
        [[1, 0, 0, 0],   <- Token 0 can only see itself
         [1, 1, 0, 0],   <- Token 1 can see tokens 0,1
         [1, 1, 1, 0],   <- Token 2 can see tokens 0,1,2
         [1, 1, 1, 1]]   <- Token 3 can see all tokens 0,1,2,3
    
    Args:
        seq_len: Sequence length
        device: Device to create mask on ('cpu' or 'cuda')
    
    Returns:
        Causal mask of shape (1, 1, seq_len, seq_len)
    """
    ### BONUS QUESTION 3.8 START

    
    ### QUESTION 3.8 END

In [ ]:
### TEST FOR BONUS QUESTION 3.8


print("=" * 60)
print("Testing Causal Mask...")
print("=" * 60)
try:
    mask = create_causal_mask(5)
    print(f"  Mask shape: {mask.shape}")
    print(f"  Mask (squeezed):")
    print(mask.squeeze().numpy())
    
    # Verify it's lower triangular
    assert mask.shape == (1, 1, 5, 5)
    expected = torch.tril(torch.ones(5, 5))
    assert torch.allclose(mask.squeeze(), expected)
    print("  ✓ Causal Mask working!")
except Exception as e:
    print(f"  ✗ Causal Mask failed: {e}")

# Test with causal mask in attention
print("\nTesting Attention with Causal Mask...")
try:
    causal_mask = create_causal_mask(seq_len)
    out_causal, attn_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
    
    # Verify upper triangle is zero (can't attend to future)
    for i in range(seq_len):
        for j in range(i + 1, seq_len):
            assert (attn_causal[:, :, i, j] == 0).all(), \
                f"Position {i} should not attend to position {j}"
    print("  ✓ Causal attention correctly masks future positions!")
except Exception as e:
    print(f"  ✗ Causal attention failed: {e}")


## End of NOAI 2026 - Programming Question 3
---